# Learning from unlabeled data: the landscape

_Semi & Self-Supervised Learning_

**Labels are scarce and pricey; raw data is cheap. Here is how to learn from the cheap stuff.**

Sort learning by what labels you have.

       
         
- Supervised: every example has a label. You learn the map from input to label directly.
         
- Semi-supervised: a small labeled set plus a large unlabeled set. You learn from both at once.
         
- Self-supervised: only unlabeled data. You invent a fake task (a "pretext" task) whose answer you can read off the data itself — predict a hidden word, match two crops of the same image — and learn useful features as a side effect.
         
- Unsupervised: only unlabeled data, and you want structure (clusters, density), not a predictor.
       

---

This notebook is a practice scaffold for the **Learning from unlabeled data: the landscape** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns** — each sample is an 8x8 grid of pixel intensities (0–16).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("image array:", digits.images.shape, " labels:", digits.target.shape)
samples = list(zip(digits.images, digits.target))
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ---- the generic semi-supervised objective: L = L_sup + lambda * L_unsup ----
# Every method in this module just fills in 'unsupervised_loss' differently.

def unsupervised_loss(model, x_unlabeled):
    """Placeholder for the method-specific term computed from UNLABELED data.

    - Consistency / FixMatch: two augmented views should predict the same class.
    - Pseudo-labeling:        train on the model's own confident guesses.
    - Contrastive (SimCLR):   two views of one image attract, others repel.
    Here we use a simple, label-free CONSISTENCY loss between two augmentations.
    """
    x1 = augment(x_unlabeled)            # weak augmentation
    x2 = augment(x_unlabeled)            # a second, independent augmentation
    p1 = F.softmax(model(x1), dim=1)
    p2 = F.softmax(model(x2), dim=1)
    # predictions on the two views should agree (symmetric squared difference)
    return ((p1 - p2) ** 2).sum(dim=1).mean()


def augment(x):
    # stand-in for real augmentations (crop, flip, color jitter, noise)
    return x + 0.05 * torch.randn_like(x)


def train_step(model, opt, labeled_batch, unlabeled_batch, lam=1.0):
    x_lab, y_lab = labeled_batch         # the few labeled examples
    x_unl, = unlabeled_batch             # the many unlabeled examples

    # 1) supervised loss on the labels we DO have
    logits = model(x_lab)
    loss_sup = F.cross_entropy(logits, y_lab)

    # 2) unsupervised loss on the unlabeled pile (no labels needed)
    loss_unsup = unsupervised_loss(model, x_unl)

    # 3) combine:  L = L_sup + lambda * L_unsup
    loss = loss_sup + lam * loss_unsup

    opt.zero_grad()
    loss.backward()
    opt.step()
    return loss_sup.item(), loss_unsup.item()


# --- usage sketch (model + two data loaders, labeled and unlabeled) ---
# model = MyNet()
# opt = torch.optim.Adam(model.parameters(), lr=1e-3)
# for labeled_batch, unlabeled_batch in zip(labeled_loader, unlabeled_loader):
#     train_step(model, opt, labeled_batch, unlabeled_batch, lam=1.0)

## Visualize the data & results

_On real handwritten digits, does adding unlabeled data beat labels-only — and does the gain shrink as labels grow plentiful?_

In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.semi_supervised import LabelSpreading
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split

digits = load_digits()                       # 1797 real 8x8 handwritten digits
X, y = digits.data / 16.0, digits.target

# fixed test set; the rest is a pool whose labels we reveal gradually
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=500, random_state=0, stratify=y)

rng = np.random.RandomState(0)
order = rng.permutation(len(y_tr))           # which pool points get labels first
label_counts = [10, 20, 30, 50, 100, 200, 400]
sup_acc, ssl_acc = [], []

for n_lab in label_counts:
    lab_idx = order[:n_lab]
    # labels-only baseline: kNN trained on just the labeled points
    knn = KNeighborsClassifier(n_neighbors=min(5, n_lab))
    knn.fit(X_tr[lab_idx], y_tr[lab_idx])
    sup_acc.append(round(float(knn.score(X_te, y_te)), 3))

    # semi-supervised: LabelSpreading over labeled + unlabeled pool (-1 = unlabeled)
    y_semi = np.full(len(y_tr), -1)
    y_semi[lab_idx] = y_tr[lab_idx]
    ls = LabelSpreading(kernel="knn", n_neighbors=7, alpha=0.2, max_iter=40)
    ls.fit(X_tr, y_semi)
    ssl_acc.append(round(float(ls.score(X_te, y_te)), 3))

print("labels :", label_counts)
print("sup    :", sup_acc)   # [0.22, 0.406, 0.532, 0.76, 0.862, 0.924, 0.964]
print("semi   :", ssl_acc)   # [0.466, 0.794, 0.902, 0.946, 0.95, 0.966, 0.972]

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** You have 1,000,000 product photos but only 200 are labeled with a category. Which learning setting is this, and what assumption lets the unlabeled photos help?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Identify the split: a tiny labeled set plus a huge unlabeled set. — _That is the definition of semi-supervised learning._
- Name the assumption: photos of the same category cluster together in feature space, and boundaries sit in the sparse gaps. — _The cluster + low-density-separation assumptions are what turn the unlabeled photos into a usable signal._

**Answer:** This is semi-supervised learning (small labeled $\mathcal{D}_L$, large unlabeled $\mathcal{D}_U$). The unlabeled photos help because of the cluster assumption — same-category photos clump together — and low-density separation — the decision boundary should fall in the empty gaps between clumps, which the million unlabeled points reveal. You would optimize $\mathcal{L}=\mathcal{L}_{\text{sup}}+\lambda\,\mathcal{L}_{\text{unsup}}$.

</details>

**Problem 2.** A teammate adds a large unlabeled set scraped from a different website and the model gets worse. Which assumption was violated?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Ask whether the unlabeled data shares the labeled data's distribution. — _Every method here assumes $\mathcal{D}_U$ is drawn from the same $p(x)$ as the task._
- A different website means different $p(x)$ — different clusters. — _The unlabeled loss then pulls the boundary toward the wrong structure._

**Answer:** The same-distribution premise behind the cluster / low-density assumptions. The scraped data comes from a different $p(x)$, so its clusters do not line up with the task's classes; $\lambda\,\mathcal{L}_{\text{unsup}}$ then drags the model toward irrelevant structure. Filter the unlabeled set to match the target domain, or lower $\lambda$.

</details>

**Problem 3.** What is the difference between self-supervised and semi-supervised learning?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Check whether any labels are used during the main learning phase. — _That is the dividing line._
- Semi-supervised uses a few real labels alongside unlabeled data; self-supervised uses none, inventing a pretext task instead. — _Self-supervised learns features first, then (optionally) fine-tunes on labels later._

**Answer:** Semi-supervised mixes a small labeled set with a large unlabeled set in one objective. Self-supervised uses only unlabeled data: it invents a pretext task whose answer is readable from the data itself (predict a masked word, match two crops of one image) and learns useful representations as a side effect — see [unl-simclr], [unl-moco], [unl-cpc] and [mod-contrastive]. Labels, if any, are spent later in a small fine-tuning step.

</details>